In [1]:
import sys
import code
import os
import pickle

import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np

from tqdm import tqdm
import scipy.sparse
from scipy.sparse import csr_matrix
from gtfparse import read_gtf
from collections import defaultdict
from sklearn.metrics import auc, precision_recall_curve, average_precision_score

sys.path.append('/home/liyang/BioWuYan/dygmamba_project/model/dygmamba/src/')
from pdata.data_read import read_unibind_file
from pdata.data_preprocess import calculate_recovery_metrics

# 1. 加载 autoreload 扩展
%load_ext autoreload

# 2. 设置模式为 "2" (表示自动重载所有模块)
%autoreload 2



In [ ]:

data_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/process/"

output_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/benchmark/"

os.makedirs(output_path, exist_ok=True )

###########################################################
# Unibind data processing

unibind_file = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/unibind/UniBind_search_qlr7ymfp.tar.gz"

unibind_df, count_region_df = read_unibind_file(unibind_file)

unibind_df["CellLine"] = unibind_df["CellLine"].str.split('_').str[0]
count_region_df["CellLine"] = count_region_df["CellLine"].str.split('_').str[0]

print(unibind_df.head())
print(count_region_df.head())

unibind_df.to_pickle(output_path + "unibind_df.pkl")
count_region_df.to_pickle(output_path + "count_region_df.pkl")




# TF recovery

In [ ]:
from analysis.assess_tf_recovery import TF_recovery_analysis

##########################################################################
# TF recovery
tf_peak_grn = ad.read_h5ad(data_path + "tf_peak_network.h5ad")

model_name = "DyGMAMBA"
TF_recovery_analysis(tf_peak_grn, count_region_df, output_path, model_name)

# TF region

In [ ]:
import sys
import code
import os
import pickle

import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np

from tqdm import tqdm
import scipy.sparse
from scipy.sparse import csr_matrix
from gtfparse import read_gtf
from collections import defaultdict
from sklearn.metrics import auc, precision_recall_curve, average_precision_score


sys.path.append('/home/liyang/BioWuYan/dygmamba_project/model/dygmamba/src/')

from pdata.data_preprocess import filter_jaspar_tf, adata_to_dataframe
from pdata.data_preprocess import build_tf_peak_network
from analysis.assess import calculate_tf_metrics



data_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/process/"

output_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/benchmark/"

unibind_df_file = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/benchmark/unibind_df.pkl"

os.makedirs(output_path, exist_ok=True )

adata_atac = ad.read_h5ad(data_path + "atac_processed.h5ad")

##################################################################
# Benchmark TF-Region
# 读取整个文件
tf_chip_seq = pd.read_parquet(data_path + "combined_chip_seq.parquet")
cell_type_chip_seq = tf_chip_seq[["chrom", "start", "end", "tf_name"]].copy()
cell_type_chip_seq.rename(columns={'tf_name':'tf'},inplace = True)
cell_type_chip_seq = cell_type_chip_seq[['tf','chrom','start','end']].copy()

unibind_df = pd.read_pickle(unibind_df_file)
unibind_chip_df = unibind_df[unibind_df['TF'].isin(set(cell_type_chip_seq["tf"]))].copy()

unibind_chip_df = unibind_chip_df[["chrom", "start", "end", "TF"]].copy()
unibind_chip_df.rename(columns={'TF':'tf'},inplace = True)
unibind_chip_df = unibind_chip_df[['tf','chrom','start','end']].copy()

unibind_tf_peak = build_tf_peak_network(adata_atac, unibind_chip_df)
unibind_tf_peak_grn = adata_to_dataframe(unibind_tf_peak)
unibind_tf_peak_grn.rename(columns={'obs':'peak_id', "var":"tf"},inplace = True)
unibind_tf_peak_grn[['chrom', 'start', 'end']] = unibind_tf_peak_grn['peak_id'].str.split('-', expand=True)
unibind_tf_peak_grn['start'] = unibind_tf_peak_grn['start'].astype(int)
unibind_tf_peak_grn['end'] = unibind_tf_peak_grn['end'].astype(int)
unibind_tf_peak_grn.rename(columns={'tf':'TF'},inplace = True)
print(unibind_tf_peak_grn)
####################################################################
# 
jaspar_tf_region_file = data_path + "jaspar_data_processed.h5ad"
jaspar_data = ad.read_h5ad(jaspar_tf_region_file)
adata_region_tf = filter_jaspar_tf(jaspar_data)

coo_matrix = adata_region_tf.X.tocoo()
tf_peak_df = pd.DataFrame({
    'Peak': adata_region_tf.obs_names[coo_matrix.row],
    'TF': adata_region_tf.var_names[coo_matrix.col],
    'value': coo_matrix.data
})
tf_peak_df = tf_peak_df[tf_peak_df["Peak"].isin(set(adata_atac.var_names))]

print("*"*50)
print(adata_region_tf)
print(tf_peak_df.head())
print(f"\n TF-Peak: {tf_peak_df['TF'].nunique()}, {tf_peak_df['Peak'].nunique()}, edge: {len(tf_peak_df)}")
print("*"*50)

###################################################################################
# 
unibind_tf_peak_grn.rename(columns={'tf':'TF'},inplace = True)

results = calculate_tf_metrics(tf_peak_df, unibind_tf_peak_grn)

print(results)

results.to_csv(output_path + "tf_region_jaspar_unibind_results.csv", index=False)






# Region-Gene

In [ ]:
import sys
import code
import os
import pickle

import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np

from tqdm import tqdm
import scipy.sparse
from scipy.sparse import csr_matrix
from gtfparse import read_gtf
from collections import defaultdict
from sklearn.metrics import auc, precision_recall_curve, average_precision_score


sys.path.append('/home/liyang/BioWuYan/dygmamba_project/model/dygmamba/src/')

from pdata.data_preprocess import adata_to_dataframe
from analysis.assess import region_gene_assess


    
data_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/process/"

output_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/benchmark/"

unibind_df_file = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/benchmark/unibind_df.pkl"

model_result_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/data_dyg/"

os.makedirs(output_path, exist_ok=True )

adata_atac = ad.read_h5ad(model_result_path + "atac.h5ad")
adata_rna = ad.read_h5ad(model_result_path + "rna.h5ad")
total_peak = set(adata_atac.var_names)

##################################################################

benchmark_peak_gene_grn = ad.read_h5ad(data_path + "peak_gene_network.h5ad")

benchmark_peak_gene_grn = benchmark_peak_gene_grn[adata_atac.var_names, adata_rna.var_names].copy()

benchmark_peak_gene_df = adata_to_dataframe(benchmark_peak_gene_grn)
benchmark_peak_gene_df = benchmark_peak_gene_df.rename(columns= {"obs":"Peak", "var":"Gene"})
print(benchmark_peak_gene_df.head())
    
############################################################################################
#

adata_rp_gene_peak = ad.read_h5ad(model_result_path + "rp_gene_peak.h5ad")
prior_peak_gene_df = adata_to_dataframe(adata_rp_gene_peak)
prior_peak_gene_df = prior_peak_gene_df.rename(columns= {"obs":"Gene", "var":"Peak"})

print("*"*50)
print(adata_rp_gene_peak)
print(prior_peak_gene_df.head())
print(f"Prior peak-gene: {prior_peak_gene_df['Peak'].nunique()}, {prior_peak_gene_df['Gene'].nunique()},\
    edge: {len(prior_peak_gene_df)}")

############################################################################################
#
    
Node_id = pd.read_pickle(model_result_path + "node_id.pkl")
graph_df = pd.read_pickle(model_result_path + "Graph_df.pkl")
graph_df["Unnamed"] = graph_df.index
name_list = ["Unnamed", "source_node", "target_node", "time", "label", "edge_idx"]
New_Graph = graph_df[name_list].copy()
New_Graph.columns = ['Unnamed: 0', 'u', 'i', 'ts', 'label', 'idx']

result_path = model_result_path + 'my_result_run0.npy'
predict_edge_label = np.load(result_path)

# binary_output = (predict_edge_label > 0.5).astype(int)
New_Graph["predict"] = predict_edge_label
predict_grn = New_Graph.copy()

mapping_series = Node_id["name"]
predict_grn['source'] = (predict_grn['u'] - 1).map(mapping_series)
predict_grn['target'] = (predict_grn['i'] - 1).map(mapping_series)

peak_gene_df = predict_grn[['source', 'target', 'ts','predict']].rename(
    columns={'source': 'Peak', 'target': 'Gene'}
)
peak_gene_df = peak_gene_df[~peak_gene_df["Gene"].str.startswith('chr')].copy()

print("*"*50)
print(peak_gene_df.head())
print(f"Peak-Gene: {peak_gene_df['Peak'].nunique()}, {peak_gene_df['Gene'].nunique()}, edge:{len(peak_gene_df)}")
print("*"*50)
    
####################################################################################

dygmamba_peak_gene_grn = peak_gene_df
avg_active_peak_gene_grn = dygmamba_peak_gene_grn.groupby(['Peak', 'Gene']).agg(
    avg_ts_weight=('predict', 'mean'),  # 对 weight 列做均值，新列名叫 avg_weight
    avg_total_weight=('predict', 'sum')  # (可选) 建议顺便算个总权重
).reset_index()
avg_active_peak_gene_grn = avg_active_peak_gene_grn[avg_active_peak_gene_grn["Peak"].isin(total_peak)].copy()
benchmark_peak_gene_df = benchmark_peak_gene_df[benchmark_peak_gene_df["Peak"].isin(total_peak)].copy()

dyg_merged_peak_gene_data = pd.merge(benchmark_peak_gene_df, avg_active_peak_gene_grn, 
                                        on = ["Gene", "Peak"], how="outer").fillna(0)

print("*"*50)
print(f"Merged Peak-Gene: {dyg_merged_peak_gene_data['Peak'].nunique()}, {dyg_merged_peak_gene_data['Gene'].nunique()}, \
    {len(dyg_merged_peak_gene_data)}")

print(f"Dygmamba Peak-Gene: {avg_active_peak_gene_grn['Peak'].nunique()}, {avg_active_peak_gene_grn['Gene'].nunique()}, \
    edge: {len(avg_active_peak_gene_grn)}")

print(f"Benchmark Peak-Gene: {benchmark_peak_gene_df['Peak'].nunique()}, \
    {benchmark_peak_gene_df['Gene'].nunique()}, edge: {len(benchmark_peak_gene_df)}")
print("*"*50)


dyg_merged_peak_gene_data["predict"] = (dyg_merged_peak_gene_data["avg_ts_weight"]>0.9).astype(int)

dyg_y_true = dyg_merged_peak_gene_data["value"].astype(int)
dyg_y_pre = dyg_merged_peak_gene_data["predict"].astype(int)
dyg_model_name = "Dygmamba_peak_gene"

result_type = "binary"
beta_value = 1
dyg_dict = dygmamba_assess(dyg_y_true, dyg_y_pre, model_name = dyg_model_name, 
                            beta = beta_value, type = result_type, fig_path = output_path)
dyg_peak_gene_result = pd.DataFrame([dyg_dict])

dyg_peak_gene_result.to_csv(output_path + "dyg_peak_gene_assess_result.csv", index=False)

print("*"*50)
print(dyg_peak_gene_result)
print(f"merged: {len(dyg_merged_peak_gene_data)}, benchmark peak gene: {len(benchmark_peak_gene_df)},\
    dyg peak gene {len(avg_active_peak_gene_grn)}")
print(f"Peak-Gene: {dyg_merged_peak_gene_data['Peak'].nunique()}, \
    {dyg_merged_peak_gene_data['Gene'].nunique()}, edge:{len(dyg_merged_peak_gene_data)}")
print("*"*50)

# TF-Gene

In [ ]:


data_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/process/"

output_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/benchmark/"

unibind_df_file = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/benchmark/unibind_df.pkl"

model_result_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/data_dyg/"

os.makedirs(output_path, exist_ok=True )

adata_atac = ad.read_h5ad(model_result_path + "atac.h5ad")
adata_rna = ad.read_h5ad(model_result_path + "rna.h5ad")
total_peak = set(adata_atac.var_names)


##################################################################
print("********************** start ********************")

start_time = time.time()  # start the time
print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Start the job")

# get arguments
args = load_link_prediction_args(is_evaluation=False)

print("**********************device********************")
print(f"Now use device is {args.device}")



########################################################################################

jaspar_tf_region_file = data_path + "jaspar_data.h5ad"
jaspar_data = ad.read_h5ad(jaspar_tf_region_file)
adata_region_tf = filter_jaspar_tf(jaspar_data)


coo_matrix = adata_region_tf.X.tocoo()
tf_peak_df = pd.DataFrame({
    'Peak': adata_region_tf.obs_names[coo_matrix.row],
    'TF': adata_region_tf.var_names[coo_matrix.col],
    'value': coo_matrix.data
})
tf_peak_df = tf_peak_df[tf_peak_df["Peak"].isin(set(adata_atac.var_names))]

print("*"*50)
print(adata_region_tf)
print(f"TF-Peak: {tf_peak_df['TF'].nunique()}, {tf_peak_df['Peak'].nunique()}, edge: {len(tf_peak_df)}")
print("*"*50)

dygmamba_tf_peak_df = tf_peak_df
dygmamba_tf_peak_df = dygmamba_tf_peak_df.rename(columns={"value":"predict"})

total_peak = set(adata_atac.var_names)
dygmamba_tf_peak_df = dygmamba_tf_peak_df[dygmamba_tf_peak_df["Peak"].isin(total_peak)].copy()

########################################################################################

feat_path = model_result_path + "edge_features.npy"
edge_label_path = model_result_path + "edge_labels.npy"

Edge_feature = np.load(feat_path, mmap_mode="r")
Edge_feature = Edge_feature.reshape(-1,1).copy()
Edge_label = np.load(edge_label_path, mmap_mode="r")
Edge_label = Edge_label.reshape(-1,1).copy()

with open(model_result_path + "node_feature_data.pkl", "rb") as f:
    load_data = pickle.load(f)

Node_feature = load_data['node_feature']

Node_id = pd.read_pickle(model_result_path + "node_id.pkl")

graph_df = pd.read_pickle(model_result_path + "Graph_df.pkl")
graph_df["Unnamed"] = graph_df.index
name_list = ["Unnamed", "source_node", "target_node", "time", "label", "edge_idx"]
New_Graph = graph_df[name_list].copy()
New_Graph.columns = ['Unnamed: 0', 'u', 'i', 'ts', 'label', 'idx']


####################################################################################

####################################################################################
result_path = model_result_path + 'my_result_run{run}.npy'
predict_edge_label = np.load(result_path)

New_Graph["predict"] = predict_edge_label
predict_grn = New_Graph.copy()

mapping_series = Node_id["name"]
predict_grn['source'] = (predict_grn['u'] - 1).map(mapping_series)
predict_grn['target'] = (predict_grn['i'] - 1).map(mapping_series)

peak_gene_df = predict_grn[['source', 'target', 'ts','predict']].rename(
    columns={'source': 'Peak', 'target': 'Gene'}
)
peak_gene_df = peak_gene_df[~peak_gene_df["Gene"].str.startswith('chr')].copy()

print("*"*50)
print(f"Peak-Gene: {peak_gene_df['Peak'].nunique()}, {peak_gene_df['Gene'].nunique()}, edge:{len(peak_gene_df)}")
print("*"*50)

#################################################################################### 
dygmamba_tf_peak_df = dygmamba_tf_peak_df.rename(columns={"predict":"value"})
peak_gene_grn = peak_gene_df[peak_gene_df["Peak"].isin(total_peak)]
merged_df = pd.merge(dygmamba_tf_peak_df, peak_gene_grn, on='Peak')

tf_gene_grn = merged_df.groupby(['TF', 'Gene', 'ts']).agg(
    peak_num=('Peak', 'nunique'),   # 对 Peak 列做去重计数，新列名叫 peak_num
    avg_weight=('predict', 'mean'),  # 对 weight 列做均值，新列名叫 avg_weight
    total_weight=('predict', 'sum')  # (可选) 建议顺便算个总权重
).reset_index()

# 查看结果
print("*"*50)
print(f"Dygmamba TF-Peak: {dygmamba_tf_peak_df['TF'].nunique()}, {dygmamba_tf_peak_df['Peak'].nunique()},\
    edge: {len(dygmamba_tf_peak_df)}")
print(f"Peak-Gene: {peak_gene_df['Peak'].nunique()}, {peak_gene_df['Gene'].nunique()}, edge:{len(peak_gene_df)}")
print(f"TF-Gene: {tf_gene_grn['TF'].nunique()}, {tf_gene_grn['Gene'].nunique()}, edge: {len(tf_gene_grn)}")

tf_gene_grn.to_pickle(model_result_path + "new_tf_gene_grn_1224.pkl")

avg_active_tf_gene_grn = tf_gene_grn.groupby(['TF', 'Gene']).agg(
    avg_ts_weight=('total_weight', 'mean'),  # 对 weight 列做均值，新列名叫 avg_weight
    avg_total_weight=('total_weight', 'sum')  # (可选) 建议顺便算个总权重
).reset_index()
avg_active_tf_gene_grn.to_pickle(model_result_path + "average_active_tf_gene_grn.pkl")

pivoted_grn = tf_gene_grn.pivot_table(
    index=['TF', 'Gene'],
    columns='ts',
    values='total_weight',
    fill_value=0
)
pivoted_grn['average_active_weight'] = pivoted_grn.mean(axis=1)
avg_global_tf_gene_grn = pivoted_grn.reset_index()
avg_global_tf_gene_grn = avg_global_tf_gene_grn[["TF","Gene","average_active_weight"]].copy()
avg_global_tf_gene_grn.columns.name = None
avg_global_tf_gene_grn.to_pickle(model_result_path + "average_global_tf_gene_grn.pkl")

print("*"*50)
print(f"TF-Gene: {tf_gene_grn['TF'].nunique()}, {tf_gene_grn['Gene'].nunique()}, edge: {len(tf_gene_grn)}")
print(f"avg active TF-Gene: {avg_active_tf_gene_grn['TF'].nunique()}, {avg_active_tf_gene_grn['Gene'].nunique()},\
    edge: {len(avg_active_tf_gene_grn)}")
print(f"avg global TF-Gene: {avg_global_tf_gene_grn['TF'].nunique()}, {avg_global_tf_gene_grn['Gene'].nunique()},\
    edge: {len(avg_global_tf_gene_grn)}")
print("*"*50)


avg_active_tf_gene_grn = pd.read_pickle(model_result_path + "average_active_tf_gene_grn.pkl")
avg_global_tf_gene_grn = pd.read_pickle(model_result_path + "average_global_tf_gene_grn.pkl")

active_grn = avg_active_tf_gene_grn[["TF","Gene"]].copy()
active_grn.rename(columns={'Gene':'Target'},inplace = True)
matrix_df = pd.DataFrame(adata_rna.X.toarray(), index=adata_rna.obs_names,
                            columns=adata_rna.var_names)

active_result, trained_models = evaluate_predictability(matrix_df, active_grn)




# Other TF-recovery

In [ ]:
import sys
import os
import pickle
import pandas as pd
import anndata as ad
import numpy as np
import networkx as nx
from tqdm import tqdm
import scipy.sparse
from sklearn.metrics import auc, precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt



# 1. 加载 autoreload 扩展
%load_ext autoreload

# 2. 设置模式为 "2" (表示自动重载所有模块)
%autoreload 2

sys.path.append('/home/liyang/BioWuYan/dygmamba_project/model/dygmamba/src/')

from analysis.assess_tf_recovery import calculate_recovery_metrics
from pdata.benchmark_data import glue_read_ctx_grn

##############################################################################
#
#
##############################################################################


In [ ]:


data_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/"

output_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/benchmark/"

os.makedirs(output_path, exist_ok=True )

###########################################################
# Unibind data processing
benchmark_df = pd.read_pickle(output_path + "count_region_df.pkl")

ground_truth_ranked = benchmark_df.sort_values(by="PeakCount", ascending=False)["TF"].tolist()

##########################################################################
# DygMamba TF recovery
tf_peak_grn = ad.read_h5ad(data_path + "process/tf_peak_network.h5ad")

dyg_tfs = set(tf_peak_grn.var_names)

dyg_x, dyg_y, dyg_raw_auc, dyg_norm_auc = calculate_recovery_metrics(ground_truth_ranked, dyg_tfs, top_n=40)

print(f"Top 40 AUC (Raw): {dyg_raw_auc:.2f}")
print(f"Top 40 AUC (Normalized): {dyg_norm_auc:.4f}")

plt.figure(figsize=(6, 6))

# 绘制我们的方法的曲线
plt.plot(dyg_x, dyg_y, label=f'{"DyGMAMBA"} (AUC={dyg_norm_auc:.2f})', color='dodgerblue', linewidth=2, marker='o', markersize=4)

##########################################################################
# GLUE TF recovery

glue_grn = glue_read_ctx_grn(data_path + "data_glue/pruned_grn.csv")


df_edges = nx.to_pandas_edgelist(glue_grn, source="TF", target="Gene")

glue_tfs = set(df_edges["TF"])

glue_x, glue_y, glue_raw_auc, glue_norm_auc = calculate_recovery_metrics(ground_truth_ranked, glue_tfs, top_n=40)

plt.plot(glue_x, glue_y, label=f'{"GLUE"} (AUC={glue_norm_auc:.2f})', color='teal', linewidth=2, marker='o', markersize=4)

##########################################################################
# CellOracle TF recovery

celloracle_grn = pd.read_csv(data_path + "data_celloracle/celloracle_results/grn_df_" + "cluster0" + ".csv")

celloracle_grn.rename(columns={"source": "TF", "target": "Gene"}, inplace= True)

oracle_tf_gene = celloracle_grn[["TF", "Gene", "-logp"]].copy()

oracle_tfs = set(oracle_tf_gene["TF"])

oracle_x, oracle_y, oracle_raw_auc, oracle_norm_auc = calculate_recovery_metrics(ground_truth_ranked, oracle_tfs, top_n=40)

plt.plot(oracle_x, oracle_y, label=f'{"CellOracle"} (AUC={oracle_norm_auc:.2f})', color='lightpink', linewidth=2, marker='o', markersize=4)

############################################################################
plt.title('TF Recovery Curve', fontsize=14)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.3)
plt.xlim(0, 40)
plt.ylim(0, 40) # 或者是实际恢复的最大值

plt.savefig(output_path + 'TF_Recovery_Curve.png', dpi=300, bbox_inches='tight')

## Tau

In [13]:
import anndata as ad

cell_cluster = ['GM12878', 'HepG2', 'K562']

adata_dict = {}

for cell_type in cell_cluster:
        
    data_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/" + cell_type + "/process"

    adata_dict[cell_type] = ad.read_h5ad(data_path + "/rna_origin.h5ad")



In [14]:
import numpy as np
import pandas as pd
import scipy.sparse
from functools import reduce

def calculate_tau_from_dict(adata_dict):
    """
    计算字典中多个 AnnData 对象之间的基因 Tau 值 (特异性指数)。
    使用通用公式：Tau = sum(1 - (x / max(x))) / (N - 1)
    
    参数:
    ----------
    adata_dict : dict
        键为细胞类型名称 (str)，值为 AnnData 对象。
        假设每个 adata.X 已经是经过标准化和 Log 转换的数据 (log1p)。
        
    返回:
    ----------
    pd.DataFrame
        包含 'Tau' 值和各细胞类型原始平均表达量的表格。
    """
    
    # 获取所有的 AnnData 对象
    adatas = list(adata_dict.values())
    keys = list(adata_dict.keys())
    n_groups = len(keys)
    
    if n_groups < 2:
        raise ValueError("计算 Tau 值至少需要 2 个或以上的细胞类型数据。")

    print(f"正在处理 {n_groups} 个细胞类型: {keys}")

    # 1. 找到所有数据集共有的基因 (Intersection)
    # 使用 reduce 函数对所有 var_names 取交集
    common_genes = reduce(np.intersect1d, [ad.var_names for ad in adatas])
    print(f"共有基因数量: {len(common_genes)}")
    
    if len(common_genes) == 0:
        return pd.DataFrame()

    # 2. 提取数据并计算平均表达量 (Pseudo-bulk aggregation)
    express_data = {}
    
    for name, adata in adata_dict.items():
        # 提取共有基因的子集
        X_subset = adata[:, common_genes].X
        
        # 计算均值 (处理稀疏矩阵)
        if scipy.sparse.issparse(X_subset):
            # mean 返回的是 matrix 类型，需要转为 array 并展平
            mean_expr = np.array(X_subset.mean(axis=0)).flatten()
        else:
            mean_expr = np.mean(X_subset, axis=0)
            
        express_data[name] = mean_expr
    
    # 构建 DataFrame (行=基因, 列=细胞类型)
    df = pd.DataFrame(express_data, index=common_genes)
    
    # 3. 计算通用 Tau 值
    # ---------------------------------------------------------
    # 公式: sum(1 - (expression / max_expression)) / (N - 1)
    # ---------------------------------------------------------
    
    # A. 获取每行的最大值 (Max expression across tissues)
    # 添加 epsilon 防止除以 0
    max_vals = df.max(axis=1)
    
    # B. 计算归一化表达量 (x_hat = x / max)
    #利用 pandas 的广播机制，每一列都除以 max_vals
    x_hat = df.div(max_vals + 1e-6, axis=0)
    
    # C. 计算求和部分: sum(1 - x_hat)
    sum_term = (1 - x_hat).sum(axis=1)
    
    # D. 最终 Tau 计算
    tau_values = sum_term / (n_groups - 1)
    
    # E. 处理全 0 的情况 (如果最大值是 0，说明所有组织都不表达，Tau 设为 NaN)
    tau_values[max_vals == 0] = np.nan
    
    # 将 Tau 结果加入 DataFrame
    df['Tau'] = tau_values
    
    # 按 Tau 值降序排列
    return df.sort_values('Tau', ascending=False)

In [ ]:
tau_df = calculate_tau_from_dict(adata_dict)

In [ ]:
print(tau_df[['Tau']].head(10)) # 只看 Tau 列

# 如果想看特定基因在各细胞系的表达情况
print(tau_df.head(5))

# 获取 Tau > 0.8 的高特异性基因
high_spec_genes = tau_df[tau_df['Tau'] > 0.8].index.tolist()
print(f"高特异性基因数量: {len(high_spec_genes)}")

# Other TF-Gene

In [44]:
from analysis.assess_tf_gene import evaluate_predictability, dyg_tf_gene_data

In [ ]:
dyg_tf_gene_data()

In [ ]:
from analysis.assess_tf_gene import evaluate_predictability

output_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/benchmark/"

dyg_result_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/data_dyg/"

os.makedirs(output_path, exist_ok=True )

adata_atac = ad.read_h5ad(dyg_result_path + "atac.h5ad")
adata_rna = ad.read_h5ad(dyg_result_path + "rna.h5ad")
matrix_df = pd.DataFrame(adata_rna.X.toarray(), index=adata_rna.obs_names,
                            columns=adata_rna.var_names)

##########################################################################
# dyg TF-Gene predictability
avg_active_tf_gene_grn = pd.read_pickle(dyg_result_path + "average_active_tf_gene_grn.pkl")
avg_global_tf_gene_grn = pd.read_pickle(dyg_result_path + "average_global_tf_gene_grn.pkl")

active_grn = avg_active_tf_gene_grn[["TF","Gene"]].copy()
active_grn.rename(columns={'Gene':'Target'},inplace = True)

active_result, trained_models = evaluate_predictability(matrix_df, active_grn)


global_grn = avg_global_tf_gene_grn[["TF","Gene"]].copy()
global_grn.rename(columns={'Gene':'Target'},inplace = True)

global_result, global_trained_models = evaluate_predictability(matrix_df, global_grn)

##########################################################################
# GLUE TF-Gene predictability

glue_grn = glue_read_ctx_grn(data_path + "data_glue/pruned_grn.csv")

glue_df = nx.to_pandas_edgelist(glue_grn, source="TF", target="Gene")

glue_df.rename(columns={'Gene':'Target'},inplace = True)
glue_result, glue_trained_models = evaluate_predictability(matrix_df, glue_df)








In [ ]:
##########################################################################
# GLUE TF-Gene predictability

glue_grn = glue_read_ctx_grn(data_path + "data_glue/pruned_grn.csv")

glue_df = nx.to_pandas_edgelist(glue_grn, source="TF", target="Gene")

glue_df.rename(columns={'Gene':'Target'},inplace = True)
glue_result, glue_trained_models = evaluate_predictability(matrix_df, glue_df)

In [ ]:
glue_result

In [ ]:
import seaborn as sns

global_result["Method"] = "Global"
active_result["Method"] = "Active"
glue_result["Method"] = "GLUE"
df_combined = pd.concat([global_result[['Correlation', 'Method']], 
                         active_result[['Correlation', 'Method']],
                         glue_result[['Correlation', 'Method']]])

plt.figure(figsize=(6, 5))

# 定义顺序 (防止自动排序乱掉)
my_order = ['Global', 'Active','GLUE']

# 4. 绘制箱形图
# Seaborn 会自动识别 'Method' 列中的组别，并独立计算每个组的箱子
ax = sns.boxplot(data=df_combined, 
                 x='Method', 
                 y='Correlation', 
                 order=my_order, 
                 palette="Set2",
                 showfliers=False) # 可选：不显示异常值点

# =================================================
# 关键技巧：修改 X 轴标签，加上 (n=xxx)
# =================================================
# 计算每组的数量
n_A = len(global_result)
n_B = len(active_result)

# 创建新的标签列表
new_labels = [f"Global \n(n={n_A})", f"Active \n(n={n_B})"]

# 应用新标签
ax.set_xticklabels(new_labels)

# 5. 美化
plt.title("TF-Gene Predictability Comparison", fontsize=14)
plt.ylabel("Correlation")
plt.xlabel("") # 清空 x 轴标题，因为标签里已经写了

sns.despine() # 去掉边框
plt.show()

In [ ]:
data_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/"

output_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/benchmark/"

dyg_result_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/data_dyg/"

os.makedirs(output_path, exist_ok=True )

adata_atac = ad.read_h5ad(dyg_result_path + "atac.h5ad")
adata_rna = ad.read_h5ad(dyg_result_path + "rna.h5ad")
matrix_df = pd.DataFrame(adata_rna.X.toarray(), index=adata_rna.obs_names,
                            columns=adata_rna.var_names)

##########################################################################
# dyg TF-Gene predictability
avg_active_tf_gene_grn = pd.read_pickle(dyg_result_path + "average_active_tf_gene_grn.pkl")
avg_global_tf_gene_grn = pd.read_pickle(dyg_result_path + "average_global_tf_gene_grn.pkl")

active_grn = avg_active_tf_gene_grn[["TF","Gene"]].copy()
active_grn.rename(columns={'Gene':'Target'},inplace = True)

active_result, trained_models = evaluate_predictability(matrix_df, active_grn)


global_grn = avg_global_tf_gene_grn[["TF","Gene"]].copy()
global_grn.rename(columns={'Gene':'Target'},inplace = True)

global_result, global_trained_models = evaluate_predictability(matrix_df, global_grn)



In [ ]:
##########################################################################
# GLUE TF-Gene predictability
from pdata.benchmark_data import glue_read_ctx_grn

glue_grn = glue_read_ctx_grn(data_path + "data_glue/pruned_grn.csv")

glue_df = nx.to_pandas_edgelist(glue_grn, source="TF", target="Gene")

glue_df.rename(columns={'Gene':'Target'},inplace = True)
glue_result, glue_trained_models = evaluate_predictability(matrix_df, glue_df)


##########################################################################
#
#
##########################################################################

global_result["Method"] = "Global"
active_result["Method"] = "Active"
glue_result["Method"] = "GLUE"
df_combined = pd.concat([global_result[['Correlation', 'Method']], 
                        active_result[['Correlation', 'Method']],
                        glue_result[['Correlation', 'Method']]])

plt.figure(figsize=(6, 5))

# 定义顺序 (防止自动排序乱掉)
my_order = ['Global', 'Active','GLUE']

# 4. 绘制箱形图
# Seaborn 会自动识别 'Method' 列中的组别，并独立计算每个组的箱子
ax = sns.boxplot(data=df_combined, 
                x='Method', 
                y='Correlation', 
                order=my_order, 
                palette="Set2",
                showfliers=False) # 可选：不显示异常值点

# =================================================
# 关键技巧：修改 X 轴标签，加上 (n=xxx)
# =================================================
# 计算每组的数量
n_A = len(global_result)
n_B = len(active_result)

# 创建新的标签列表
new_labels = [f"Global \n(n={n_A})", f"Active \n(n={n_B})"]

# 应用新标签
ax.set_xticklabels(new_labels)

# 5. 美化
plt.title("TF-Gene Predictability Comparison", fontsize=14)
plt.ylabel("Correlation")
plt.xlabel("") # 清空 x 轴标题，因为标签里已经写了

sns.despine() # 去掉边框

plt.savefig(output_path + "TF_Gene_boxplot_comparison.png", dpi=300, bbox_inches='tight')



# Other TF-Region

In [ ]:
import sys
import code
import os
import pickle

import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np

from tqdm import tqdm
import scipy.sparse
from scipy.sparse import csr_matrix
from gtfparse import read_gtf
from collections import defaultdict
from sklearn.metrics import auc, precision_recall_curve, average_precision_score
import dill
import networkx as nx
# 1. 加载 autoreload 扩展
%load_ext autoreload

# 2. 设置模式为 "2" (表示自动重载所有模块)
%autoreload 2


sys.path.append('/home/liyang/BioWuYan/dygmamba_project/model/dygmamba/src/')

from pdata.data_preprocess import filter_jaspar_tf, adata_to_dataframe
from pdata.data_preprocess import build_tf_peak_network
from analysis.assess_tf_region import calculate_tf_metrics


In [ ]:

data_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/process/"

output_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/benchmark/"

unibind_df_file = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/benchmark/unibind_df.pkl"

os.makedirs(output_path, exist_ok=True )

adata_atac = ad.read_h5ad(data_path + "atac_processed.h5ad")

##################################################################
# Benchmark TF-Region
# 读取整个文件
tf_chip_seq = pd.read_parquet(data_path + "combined_chip_seq.parquet")
cell_type_chip_seq = tf_chip_seq[["chrom", "start", "end", "tf_name"]].copy()
cell_type_chip_seq.rename(columns={'tf_name':'tf'},inplace = True)
cell_type_chip_seq = cell_type_chip_seq[['tf','chrom','start','end']].copy()

unibind_df = pd.read_pickle(unibind_df_file)
unibind_chip_df = unibind_df[unibind_df['TF'].isin(set(cell_type_chip_seq["tf"]))].copy()

unibind_chip_df = unibind_chip_df[["chrom", "start", "end", "TF"]].copy()
unibind_chip_df.rename(columns={'TF':'tf'},inplace = True)
unibind_chip_df = unibind_chip_df[['tf','chrom','start','end']].copy()

unibind_tf_peak = build_tf_peak_network(adata_atac, unibind_chip_df)
unibind_tf_peak_grn = adata_to_dataframe(unibind_tf_peak)
unibind_tf_peak_grn.rename(columns={'obs':'peak_id', "var":"tf"},inplace = True)
unibind_tf_peak_grn[['chrom', 'start', 'end']] = unibind_tf_peak_grn['peak_id'].str.split('-', expand=True)
unibind_tf_peak_grn['start'] = unibind_tf_peak_grn['start'].astype(int)
unibind_tf_peak_grn['end'] = unibind_tf_peak_grn['end'].astype(int)
unibind_tf_peak_grn.rename(columns={'tf':'TF'},inplace = True)

####################################################################
# dyg TF-Region
jaspar_tf_region_file = data_path + "jaspar_data_processed.h5ad"
jaspar_data = ad.read_h5ad(jaspar_tf_region_file)
adata_region_tf = filter_jaspar_tf(jaspar_data)

coo_matrix = adata_region_tf.X.tocoo()
tf_peak_df = pd.DataFrame({
    'Peak': adata_region_tf.obs_names[coo_matrix.row],
    'TF': adata_region_tf.var_names[coo_matrix.col],
    'value': coo_matrix.data
})
tf_peak_df = tf_peak_df[tf_peak_df["Peak"].isin(set(adata_atac.var_names))]

dyg_results = calculate_tf_metrics(tf_peak_df, unibind_tf_peak_grn)

# print(results)

# results.to_csv(output_path + "tf_region_jaspar_unibind_results.csv", index=False)






In [ ]:
dyg_results

In [ ]:
import dill

glue_data_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/data_glue/"
with open(glue_data_path + "peak2tf.pkl", "rb") as f:
    tf_peak_nx = dill.load(f)
glue_tf_peak_df = nx.to_pandas_edgelist(tf_peak_nx) 
glue_tf_peak_df["value"] = 1
glue_tf_peak_df.rename(columns={'source':'Peak', 'target':'TF'}, inplace=True)

glue_result = calculate_tf_metrics(glue_tf_peak_df, unibind_tf_peak_grn)
glue_result

In [ ]:
dyg_results = calculate_tf_metrics(tf_peak_df, unibind_tf_peak_grn)

print("*"*20 + " DYGMAMBA " + "*"*20)
print(dyg_results)

# results.to_csv(output_path + "tf_region_jaspar_unibind_results.csv", index=False)
####################################################################
# GLUE TF-Region
glue_data_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/data_glue/"
with open(glue_data_path + "peak2tf.pkl", "rb") as f:
    tf_peak_nx = dill.load(f)
glue_tf_peak_df = nx.to_pandas_edgelist(tf_peak_nx) 
glue_tf_peak_df["value"] = 1
glue_tf_peak_df.rename(columns={'source':'Peak', 'target':'TF'}, inplace=True)

glue_results = calculate_tf_metrics(glue_tf_peak_df, unibind_tf_peak_grn)
print("*"*20 + " GLUE " + "*"*20)
print(glue_results)

In [ ]:
import seaborn as sns

dyg_results['Method'] = 'DYG'
glue_results['Method'] = 'GLUE'

# 拼接在一起 (只取 F1 和 Method 列)
df_combined = pd.concat([dyg_results[['F1', 'Method']], 
                         glue_results[['F1', 'Method']]])

# ==========================================
# 3. 绘制箱形图
# ==========================================
# 设置绘图风格
sns.set_theme(style="ticks")
plt.figure(figsize=(6, 5))

# 画图
ax = sns.boxplot(data=df_combined, 
                 x='Method',    # x轴放分组标签
                 y='F1',        # y轴放数值
                 palette="Set2", # 配色方案
                 width=0.5)     # 箱子宽度

# (可选) 添加抖动散点，展示原始数据点分布
sns.stripplot(data=df_combined, x='Method', y='F1', 
              color='black', size=3, alpha=0.5, jitter=True)

# ==========================================
# 4. 美化与保存
# ==========================================
plt.title("F1 Score Comparison")
plt.ylabel("F1 Score")
plt.xlabel("") # 去掉 x 轴的 'Method' 字样，因为标签已经很清楚了
sns.despine()  # 去掉上方和右侧的边框，更符合学术规范

# 保存图片
# plt.savefig("f1_boxplot_comparison.pdf", bbox_inches='tight')
plt.show()

# Other Region-Gene

In [ ]:
import sys
import code
import os
import pickle

import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np

from tqdm import tqdm
import scipy.sparse
from scipy.sparse import csr_matrix
from gtfparse import read_gtf
from collections import defaultdict
from sklearn.metrics import auc, precision_recall_curve, average_precision_score

sys.path.append('/home/liyang/BioWuYan/dygmamba_project/model/dygmamba/src/')


from pdata.data_preprocess import adata_to_dataframe
from benchmark.benchmark_process import Region_Gene_Benchmark


# 1. 加载 autoreload 扩展
%load_ext autoreload

# 2. 设置模式为 "2" (表示自动重载所有模块)
%autoreload 2


    

In [ ]:
Region_Gene_Benchmark()

In [ ]:
from analysis.assess_region_gene import region_gene_evaluate, evaluate_per_gene_correlation, evaluate_scenic_plus_correlation
    
data_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/process/"

output_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/benchmark/"

unibind_df_file = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/benchmark/unibind_df.pkl"

model_result_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/data_dyg/"

os.makedirs(output_path, exist_ok=True)

adata_atac = ad.read_h5ad(model_result_path + "atac.h5ad")
adata_rna = ad.read_h5ad(model_result_path + "rna.h5ad")
total_peak = set(adata_atac.var_names)

##################################################################

benchmark_peak_gene_df = pd.read_pickle(data_path + "peak_gene_df.pkl")
benchmark_peak_gene_df = benchmark_peak_gene_df.rename(columns= {"PeakID":"Peak", "gene_name":"Gene","hic_score":"value"})
benchmark_peak_gene_df["label"] = (benchmark_peak_gene_df["value"]>0).astype(int)

print(benchmark_peak_gene_df.head())

Markrer_Genes = adata_rna.var["highly_variable_rank"].copy()
Markrer_Genes = Markrer_Genes.sort_values()
top_100_marker_genes = list(Markrer_Genes.index[:100])
    
############################################################################################
#

adata_rp_gene_peak = ad.read_h5ad(model_result_path + "rp_gene_peak.h5ad")
prior_peak_gene_df = adata_to_dataframe(adata_rp_gene_peak)
prior_peak_gene_df = prior_peak_gene_df.rename(columns= {"obs":"Gene", "var":"Peak"})

print("*"*50)
print(adata_rp_gene_peak)
print(prior_peak_gene_df.head())
print(f"Prior peak-gene: {prior_peak_gene_df['Peak'].nunique()}, {prior_peak_gene_df['Gene'].nunique()},\
    edge: {len(prior_peak_gene_df)}")

############################################################################################
# dyg Region-Gene prediction
    
Node_id = pd.read_pickle(model_result_path + "node_id.pkl")
graph_df = pd.read_pickle(model_result_path + "Graph_df.pkl")
graph_df["Unnamed"] = graph_df.index
name_list = ["Unnamed", "source_node", "target_node", "time", "label", "edge_idx"]
New_Graph = graph_df[name_list].copy()
New_Graph.columns = ['Unnamed: 0', 'u', 'i', 'ts', 'label', 'idx']

result_path = model_result_path + 'my_result_run0.npy'
predict_edge_label = np.load(result_path)

# binary_output = (predict_edge_label > 0.5).astype(int)
New_Graph["predict"] = predict_edge_label
predict_grn = New_Graph.copy()

mapping_series = Node_id["name"]
predict_grn['source'] = (predict_grn['u'] - 1).map(mapping_series)
predict_grn['target'] = (predict_grn['i'] - 1).map(mapping_series)

peak_gene_df = predict_grn[['source', 'target', 'ts','predict']].rename(
    columns={'source': 'Peak', 'target': 'Gene'}
)
peak_gene_df = peak_gene_df[~peak_gene_df["Gene"].str.startswith('chr')].copy()

print("*"*50)
print(peak_gene_df.head())
print(f"Peak-Gene: {peak_gene_df['Peak'].nunique()}, {peak_gene_df['Gene'].nunique()}, edge:{len(peak_gene_df)}")
print("*"*50)
    
####################################################################################

dygmamba_peak_gene_grn = peak_gene_df
avg_active_peak_gene_grn = dygmamba_peak_gene_grn.groupby(['Peak', 'Gene']).agg(
    avg_ts_weight=('predict', 'mean'),  # 对 weight 列做均值，新列名叫 avg_weight
    avg_total_weight=('predict', 'sum')  # (可选) 建议顺便算个总权重
).reset_index()
avg_active_peak_gene_grn = avg_active_peak_gene_grn[avg_active_peak_gene_grn["Peak"].isin(total_peak)].copy()

benchmark_peak_gene_df = benchmark_peak_gene_df[benchmark_peak_gene_df["Peak"].isin(total_peak)].copy()

avg_active_peak_gene_grn["predict"] = (avg_active_peak_gene_grn["avg_ts_weight"]> 0.9).astype(int)

dyg_peak_gene_result = region_gene_evaluate(benchmark_peak_gene_df, avg_active_peak_gene_grn, 
                                            model_name="DYGMAMBA", output_path=output_path)


#######################################################################################
#
#
#######################################################################################



df_hic = benchmark_peak_gene_df.copy()
df_hic = df_hic.rename(columns={"Peak":"region", "Gene":"gene"})

df_pred = avg_active_peak_gene_grn.copy()
df_pred = df_pred.rename(columns={"Peak":"region", "Gene":"gene", "avg_ts_weight":"value"})

results = evaluate_per_gene_correlation(df_pred, df_hic, marker_genes=top_100_marker_genes, min_links=3)

#######################################################################################
#
#
#######################################################################################

corr_score, plot_data = evaluate_scenic_plus_correlation(
    df_pred, df_hic, marker_genes=top_100_marker_genes)

print("*"*50)
print(corr_score)
print("*"*50)
print(results)
print("*"*50)
print(dyg_peak_gene_result)

In [ ]:
from analysis.assess_region_gene import evaluate_per_gene_correlation, evaluate_scenic_plus_correlation
from analysis.assess_region_gene import region_gene_evaluate

data_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/process/"

output_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/benchmark/"

unibind_df_file = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/benchmark/unibind_df.pkl"

model_result_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/data_dyg/"

os.makedirs(output_path, exist_ok=True )

adata_atac = ad.read_h5ad(model_result_path + "atac.h5ad")
adata_rna = ad.read_h5ad(model_result_path + "rna.h5ad")
total_peak = set(adata_atac.var_names)
total_gene = set(adata_rna.var_names)

##################################################################
benchmark_peak_gene_df = pd.read_pickle(data_path + "peak_gene_df.pkl")
benchmark_peak_gene_df = benchmark_peak_gene_df.rename(columns= {"PeakID":"Peak", "gene_name":"Gene","hic_score":"value"})
benchmark_peak_gene_df["label"] = (benchmark_peak_gene_df["value"]>0).astype(int)
benchmark_peak_gene_df = benchmark_peak_gene_df[benchmark_peak_gene_df["Peak"].isin(total_peak)].copy()
benchmark_peak_gene_df = benchmark_peak_gene_df[benchmark_peak_gene_df["Gene"].isin(total_gene)].copy()

df_hic = benchmark_peak_gene_df.copy()
df_hic = df_hic.rename(columns={"Peak":"region", "Gene":"gene"})

############################################################################################

Markrer_Genes = adata_rna.var["highly_variable_rank"].copy()
Markrer_Genes = Markrer_Genes.sort_values()
top_marker_genes = list(Markrer_Genes.index[:100])

############################################################################################
# DYGMAMBA peak-gene grn
    
Node_id = pd.read_pickle(model_result_path + "node_id.pkl")
graph_df = pd.read_pickle(model_result_path + "Graph_df.pkl")
graph_df["Unnamed"] = graph_df.index
name_list = ["Unnamed", "source_node", "target_node", "time", "label", "edge_idx"]
New_Graph = graph_df[name_list].copy()
New_Graph.columns = ['Unnamed: 0', 'u', 'i', 'ts', 'label', 'idx']

result_path = model_result_path + 'my_result_run0.npy'
predict_edge_label = np.load(result_path)

# binary_output = (predict_edge_label > 0.5).astype(int)
New_Graph["predict"] = predict_edge_label
dyg_predict_grn = New_Graph.copy()

mapping_series = Node_id["name"]
dyg_predict_grn['source'] = (dyg_predict_grn['u'] - 1).map(mapping_series)
dyg_predict_grn['target'] = (dyg_predict_grn['i'] - 1).map(mapping_series)

dyg_peak_gene_df = dyg_predict_grn[['source', 'target', 'ts','predict']].rename(
    columns={'source': 'Peak', 'target': 'Gene'}
)
dyg_peak_gene_df = dyg_peak_gene_df[~dyg_peak_gene_df["Gene"].str.startswith('chr')].copy()
    
####################################################################################

avg_active_peak_gene_grn = dyg_peak_gene_df.groupby(['Peak', 'Gene']).agg(
    avg_ts_weight=('predict', 'mean'),  # 对 weight 列做均值，新列名叫 avg_weight
    avg_total_weight=('predict', 'sum')  # (可选) 建议顺便算个总权重
).reset_index()
avg_active_peak_gene_grn = avg_active_peak_gene_grn[avg_active_peak_gene_grn["Peak"].isin(total_peak)].copy()


avg_active_peak_gene_grn["predict"] = (avg_active_peak_gene_grn["avg_ts_weight"]> 0.9).astype(int)

dyg_peak_gene_result = region_gene_evaluate(benchmark_peak_gene_df, avg_active_peak_gene_grn, 
                                            model_name="DYGMAMBA", output_path=output_path)

df_pred = avg_active_peak_gene_grn.copy()
df_pred = df_pred.rename(columns={"Peak":"region", "Gene":"gene", "avg_ts_weight":"value"})

dyg_results = evaluate_per_gene_correlation(df_pred, df_hic, marker_genes=top_marker_genes, min_links=3)

dyg_corr_score, dyg_plot_data = evaluate_scenic_plus_correlation(df_pred, df_hic, 
                                                            marker_genes=top_marker_genes)
print("*"*50)
print("*"*20 + " DYGMAMBA " + "*"*20)
print(dyg_corr_score)
print("*"*50)
print(dyg_results)
print("*"*50)
print(dyg_peak_gene_result)
#######################################################################################
#
#
#######################################################################################
# GLUE peak-gene grn

glue_data_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/data_glue/"

with open(glue_data_path + "gene2peak.pkl", "rb") as f:
    gene_peak_nx = dill.load(f)
glue_gene_peak_df = nx.to_pandas_edgelist(gene_peak_nx)

glue_gene_peak_df.rename(columns={'source':'Gene', 'target':'Peak', 'weight':'predict'}, inplace=True)
glue_gene_peak_df["predict"] = (glue_gene_peak_df["predict"]>0.9).astype(int)
glue_gene_peak_df["value"] = glue_gene_peak_df["score"]
glue_pred = glue_gene_peak_df[["Peak", "Gene", "value"]].copy()
glue_pred = glue_pred.rename(columns={"Peak":"region", "Gene":"gene"})

#######################################################################################
#
#
#######################################################################################

glue_peak_gene_result = region_gene_evaluate(benchmark_peak_gene_df, glue_gene_peak_df, 
                                            model_name="GLUE", output_path=output_path)  

glue_results = evaluate_per_gene_correlation(glue_pred, df_hic, marker_genes=top_marker_genes, min_links=3)

glue_corr_score, glue_plot_data = evaluate_scenic_plus_correlation(glue_pred, df_hic, 
                                                                    marker_genes=top_marker_genes)

print("*"*50)
print("*"*20 + " GLUE " + "*"*20)
print(glue_corr_score)
print("*"*50)
print(glue_results)
print("*"*50)
print(glue_peak_gene_result)

In [ ]:
################
dyg_results['Method'] = 'DYG'
glue_results['Method'] = 'GLUE'
analy_columns = ['correlation', 'Method']
x_axis_label = "Method"
y_axis_label = "correlation"
# 拼接在一起 (只取 F1 和 Method 列)
df_combined = pd.concat([dyg_results[analy_columns], 
                        glue_results[analy_columns]])

# ==========================================
# 3. 绘制箱形图
# ==========================================
# 设置绘图风格
sns.set_theme(style="ticks")
plt.figure(figsize=(6, 5))

# 画图
ax = sns.boxplot(data=df_combined, 
                x=x_axis_label,    # x轴放分组标签
                y=y_axis_label,        # y轴放数值
                palette="Set2", # 配色方案
                width=0.5)     # 箱子宽度

# (可选) 添加抖动散点，展示原始数据点分布
sns.stripplot(data=df_combined, x=x_axis_label, y=y_axis_label, 
            color='black', size=3, alpha=0.5, jitter=True)

# ==========================================
# 4. 美化与保存
# ==========================================
plt.title(y_axis_label + " Score Comparison")
plt.ylabel(y_axis_label + " Score")
plt.xlabel("") # 去掉 x 轴的 'Method' 字样，因为标签已经很清楚了
sns.despine()  # 去掉上方和右侧的边框，更符合学术规范

# 保存图片
# plt.savefig("f1_boxplot_comparison.pdf", bbox_inches='tight')
plt.show()

## GLUE

In [ ]:
import dill

glue_data_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/data_glue/"

with open(glue_data_path + "gene2peak.pkl", "rb") as f:
    gene_peak_nx = dill.load(f)
    
with open(glue_data_path + "peak2tf.pkl", "rb") as f:
    tf_peak_nx = dill.load(f)
    

glue_gene_peak_df = nx.to_pandas_edgelist(gene_peak_nx)

glue_tf_peak_df = nx.to_pandas_edgelist(tf_peak_nx) 

glue_gene_peak_df.rename(columns={'source':'Gene', 'target':'Peak', 'weight':'predict'}, inplace=True)
glue_gene_peak_df["predict"] = (glue_gene_peak_df["predict"]>0.9).astype(int)

#######################################################################################
#
#
#######################################################################################

glue_peak_gene_result = region_gene_evaluate(benchmark_peak_gene_df, glue_gene_peak_df, 
                                            model_name="GLUE", output_path=output_path)  
#######################################################################################
#
#
#######################################################################################

glue_gene_peak_df["value"] = glue_gene_peak_df["score"]
glue_pred = glue_gene_peak_df[["Peak", "Gene", "value"]].copy()
glue_pred = glue_pred.rename(columns={"Peak":"region", "Gene":"gene"})

glue_results = evaluate_per_gene_correlation(glue_pred, df_hic, marker_genes=top_100_marker_genes, min_links=3)

#######################################################################################
#
#
#######################################################################################

glue_corr_score, glue_plot_data = evaluate_scenic_plus_correlation(
    df_pred, df_hic, marker_genes=top_100_marker_genes)

print("*"*50)
print(glue_corr_score)
print("*"*50)
print(glue_results)
print("*"*50)
print(glue_peak_gene_result)

In [ ]:

glue_gene_peak_df


In [ ]:
glue_pred


In [ ]:
print(f"glue peak: {len(set(glue_gene_peak_df['Peak']))}")

print(f"benchmark peak: {len(set(benchmark_peak_gene_df['Peak']))}")

print(f"dyg peak: {len(set(avg_active_peak_gene_grn['Peak']))}")


print(f"atac peak: {len(set(adata_atac.var_names))}")

print(f"glue & benchmark: {len(set(glue_gene_peak_df['Peak']) & set(benchmark_peak_gene_df['Peak']))}")

print(f"glue & atac: {len(set(glue_gene_peak_df['Peak']) & set(adata_atac.var_names))}")

print(f"benchmark & atac: {len(set(benchmark_peak_gene_df['Peak']) & set(adata_atac.var_names))}")

print(f"dyg & atac: {len(set(avg_active_peak_gene_grn['Peak']) & set(adata_atac.var_names))}")




## TEST


In [ ]:
from scipy import stats

marker_genes = top_100_marker_genes
method = 'spearman'


pred = df_pred.rename(columns={'value': 'pred_score'})
hic = df_hic.rename(columns={'value': 'hic_score'})

# 确保 key 列是字符串类型，防止因类型不一致导致 merge 失败
for df in [pred, hic]:
    df['region'] = df['region'].astype(str)
    df['gene'] = df['gene'].astype(str)

# 2. (关键步骤) 筛选 Marker Genes
# SCENIC+ 原文："for the top 100 marker genes... correlations were calculated"
if marker_genes is not None:
    print(f"正在筛选 {len(marker_genes)} 个 Marker Genes...")
    pred = pred[pred['gene'].isin(marker_genes)]
    # Hi-C 数据通常很大，先过滤可以加速 merge
    hic = hic[hic['gene'].isin(marker_genes)]
    
    if pred.empty:
        print("警告：筛选后预测结果为空！请检查 Marker Genes 名字是否与 dataframe 一致。")

# 3. (关键步骤) 数据对齐 - Inner Join (交集)
# 只有同时存在于预测和 Hi-C 中的边才参与相关性计算
print("正在合并预测数据与 Hi-C 数据...")
merged_df = pd.merge(pred, hic, on=['region', 'gene'], how='inner')

n_links = len(merged_df)
print(f"共找到 {n_links} 个重叠的 Region-Gene 连接用于评估。")

if n_links < 10:
    print("警告：重叠连接数过少，相关性计算可能不可靠。")

# 4. 计算相关性
# Spearman 关注的是“排名”：预测分越高的，是不是 Hi-C 分也越高？
# 这比 Pearson 更适合，因为 Hi-C 数据通常不服从正态分布
if method == 'spearman':
    corr, p_val = stats.spearmanr(merged_df['pred_score'], merged_df['hic_score'])
else:
    corr, p_val = stats.pearsonr(merged_df['pred_score'], merged_df['hic_score'])
    
print(f"评估结果: Correlation = {corr:.4f} (P-value = {p_val:.2e})")


In [ ]:
merged_df['pred_score']
merged_df['hic_score']

# Other 

In [ ]:
data_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/process/"

output_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/benchmark/"

unibind_df_file = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/benchmark/unibind_df.pkl"

model_result_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/data_dyg/"

os.makedirs(output_path, exist_ok=True )

adata_atac = ad.read_h5ad(model_result_path + "atac.h5ad")
adata_rna = ad.read_h5ad(model_result_path + "rna.h5ad")
total_peak = set(adata_atac.var_names)

##################################################################

benchmark_peak_gene_grn = ad.read_h5ad(data_path + "peak_gene_network.h5ad")

benchmark_peak_gene_grn = benchmark_peak_gene_grn[list(adata_atac.var_names), list(adata_rna.var_names)].copy()

In [ ]:
benchmark_peak_gene_grn

In [5]:
import pandas as pd
chip_seq_path = "/home/liyang/BioWuYan/dygmamba_project/data/TF_ChIP_seq/code/combined_chip_seq.parquet"
tf_chip_seq_scenic = pd.read_parquet(chip_seq_path)
    
# tf_chip_seq_scenic = tf_chip_seq_scenic[tf_chip_seq_scenic["cell_type"] == cell_type]

In [ ]:
set(tf_chip_seq_scenic["cell_type"])

In [9]:
cell_cluster = ['GM12878', 'HepG2', 'IMR90', 'K562', 'MCF7', 'PC3', 'Panc1']

In [ ]:
"GM12878" in cell_cluster

In [1]:
import anndata as ad
rp_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/process/binary_peak_gene_rp_network.h5ad"

adata_rp = ad.read_h5ad(rp_path)

In [ ]:
adata_rp

In [ ]:
import numpy as np
import scipy.sparse

cell_type = "GM12878"
output_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/" + cell_type + "/process/"
adata_atac = ad.read_h5ad(output_path + "atac_origin.h5ad")
adata_atac